In [ ]:
# !python -c "import numpy; import pandas; print(numpy.__version__, pandas.__version__)"


In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("Is built with CUDA:", tf.test.is_built_with_cuda())
print("GPU devices found:", tf.config.list_physical_devices('GPU'))


In [ ]:
import argparse
from data_loader import load_and_preprocess_data
from rnn_model import create_rnn_model, train_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

parser = argparse.ArgumentParser()
# Setting default argument to 19 for 19 Class multiclassification
parser.add_argument('--class_config', type=int, choices=[2, 6, 19], default=19)
args, unknown = parser.parse_known_args()

data_path = "../data"

X_train, X_val, X_test, y_train_categorical, y_val_categorical, y_test_categorical, label_encoder = load_and_preprocess_data(
    data_path,
    args.class_config
)

input_shape = (X_train.shape[1], X_train.shape[2])
num_classes = y_train_categorical.shape[1]

model = create_rnn_model(input_shape, num_classes)


# Define unique filepath with epoch placeholder
checkpoint_path = "rnn_model_epoch_{epoch:02d}.h5"

# Create callback
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=False,  # Set to True to save only weights
    save_best_only=False,     # Save every epoch
    verbose=1
)
print("Callbacks set up. Starting training...")


model = train_model(
    model,
    X_train,
    y_train_categorical,
    X_val,
    y_val_categorical
)

loss, accuracy = model.evaluate(X_test, y_test_categorical)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

y_pred_categorical = model.predict(X_test)
y_pred_encoded = y_pred_categorical.argmax(axis=1)
y_pred = label_encoder.inverse_transform(y_pred_encoded)

y_test_decoded = label_encoder.inverse_transform(y_test_categorical.argmax(axis=1))



accuracy = accuracy_score(y_test_decoded, y_pred)
precision = precision_score(y_test_decoded, y_pred, average='weighted')
recall = recall_score(y_test_decoded, y_pred, average='weighted')
f1 = f1_score(y_test_decoded, y_pred, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("\nClassification Report:\n", classification_report(y_test_decoded, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_decoded, y_pred))


d:\ProgramData\anaconda3\envs\CS4371\lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
usage: ipykernel_launcher.py [-h] [--class_config {2,6,19}]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\boboh\AppData\Roaming\jupyter\runtime\kernel-v3ffb7fe423cff759957a094af2921ed267101b075.json


SystemExit: 2

d:\ProgramData\anaconda3\envs\CS4371\lib\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
